### Unet parts

In [2]:
import torch
import torch.nn as nn
import torch.functional as F

In [ ]:
class DoubleConv(nn.Module):

    def __init__(self , in_channels , out_channels , mid_channels = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.sequential(
            nn.Conv2d(in_channels , mid_channels , kernel_size = 3 , padding = 1 , bias = False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace = True),
            nn.Conv2d(mid_channels , out_channels , kernel_size = 3 , padding = 1 , bias = False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True)
        )
    def forward(self ,x):
        return self.max_conv(x)

#Down sampling the output images

class Down(nn.Module):

    '''downscaling with maxpool then double conv'''

    def __init__(self , in_channels , out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels , out_channels)
        )            

    def forward(self,x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self  , in_channels , out_channels , bilinear = True):
        super().__init__()

        if bilinear:
            self.up = nn.Upsample(scale_factor = 2 ,mode = 'bilinear' , align_corners = True)
            self.conv = DoubleConv(in_channels , out_channels ,in_channels//2)

        else:
            self.up = nn.ConvTranpose2d(in_channels , in_channels//2 , kernel_size = 2 , stride = 2)
            self.conv = DoubleConv(in_channels , out_channels)
